# Independent review of PR #46
Mia, Yassin, or Nastassja: open this notebook in Google Colab, use a standard CPU runtime, enter your own name below, and run all cells. This runs in a remote Linux environment rather than on a Windows machine whose App Control blocks pandas. Use an environment permitted by your institution; this notebook does not change device security policies.
The notebook reads the current PR head, checks out that exact commit, runs the six synthetic tests, obtains the team's public Kaggle replication, verifies its five checksums, and rebuilds the real split twice. It does not train the model or approve/merge any PR.
A successful run must show six passing tests and 95,120 training rows, 23,781 testing rows, and 27 features. Download the evidence ZIP from the final cell and attach it to PR #46 with your own confirmation. An author/AI run does not replace a teammate's independent run. Source-code execution is expected when you run this project.


In [ ]:
REVIEWER = 'ENTER YOUR NAME'  # Replace this with the person actually running the notebook.
assert REVIEWER != 'ENTER YOUR NAME', 'Enter your own reviewer name before running.'
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', 'numpy', 'pandas', 'scikit-learn', 'kagglehub'], check=True)


In [ ]:
from pathlib import Path
import json, urllib.request, datetime, platform, importlib.metadata, re, shutil
request = urllib.request.Request('https://api.github.com/repos/n4stam0on/Intrusion-Interceptors-Pipeline-Capstone/pulls/46', headers={'User-Agent': 'capstone-independent-review'})
with urllib.request.urlopen(request, timeout=60) as response:
    pr = json.load(response)
reviewed_commit = pr['head']['sha']
REPO = Path.cwd() / ('pr46-review-' + reviewed_commit[:12])
if REPO.exists():
    raise RuntimeError('This checkout already exists. Start a fresh runtime or use a new directory; do not overwrite earlier evidence.')
subprocess.run(['git', 'clone', '--no-checkout', 'https://github.com/n4stam0on/Intrusion-Interceptors-Pipeline-Capstone.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', reviewed_commit], check=True)
EVIDENCE = REPO / 'data/processed/independent-review'
EVIDENCE.mkdir(parents=True)
print('Reviewer:', REVIEWER, '\nExact commit:', reviewed_commit, '\nPython:', platform.python_version(), '\nOS:', platform.system())


In [ ]:
tests = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=REPO, capture_output=True, text=True)
unit_output = tests.stdout + tests.stderr
print(unit_output)
(EVIDENCE / 'unit-tests.txt').write_text(unit_output, encoding='utf-8')
assert tests.returncode == 0, 'Tests failed: stop and share the error.'
assert re.search(r'Ran 6 tests?', unit_output), 'Expected six tests; review the changed test inventory.'


In [ ]:
import kagglehub
download_root = Path(kagglehub.dataset_download('ogunyemioluwapelumi/mqtt-iot-ids2020-private'))
input_dir = REPO / 'data/raw/kaggle-replication'
input_dir.mkdir(parents=True)
for name in ['biflow_normal.csv', 'biflow_scan_A.csv', 'biflow_scan_sU.csv', 'biflow_sparta.csv', 'biflow_mqtt_bruteforce.csv']:
    candidates = list(download_root.rglob(name))
    assert len(candidates) == 1, f'Expected exactly one {name}, found {len(candidates)}'
    shutil.copyfile(candidates[0], input_dir / name)
command = [sys.executable, 'src/verify_prepared_data.py', '--input-dir', str(input_dir), '--output-dir', str(EVIDENCE / 'runs'), '--report', str(EVIDENCE / 'data-verification.json')]
verification = subprocess.run(command, cwd=REPO, capture_output=True, text=True)
output = verification.stdout + verification.stderr
print(output)
(EVIDENCE / 'data-verification-output.txt').write_text(output, encoding='utf-8')
assert verification.returncode == 0, 'Real-data verification failed: stop and share the error.'
report = json.loads((EVIDENCE / 'data-verification.json').read_text())
m = report['metadata']
assert (m['train_rows'], m['test_rows'], m['feature_count']) == (95120, 23781, 27)
baseline = json.loads((REPO / 'results/data-verification.json').read_text())
hash_comparison = {key: value == baseline['logical_array_sha256'][key] for key, value in report['logical_array_sha256'].items()}
print('PASS: 6 tests; 95,120 train / 23,781 test / 27 features')
print('Exact baseline hash comparison:', hash_comparison)
print('Different platform/package versions can change float hashes. Disclose differences; do not claim exact hash equality unless every entry is True.')


In [ ]:
summary = {'reviewer': REVIEWER, 'executed_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'reviewed_commit': reviewed_commit, 'platform': platform.system(), 'python': platform.python_version(), 'packages': {name: importlib.metadata.version(name) for name in ['numpy', 'pandas', 'scikit-learn', 'kagglehub']}, 'unit_tests': '6 passed', 'train_rows': m['train_rows'], 'test_rows': m['test_rows'], 'feature_count': m['feature_count'], 'baseline_hash_match': hash_comparison, 'scope': 'Public Kaggle replication; not official-archive equivalence or independent-session generalization'}
(EVIDENCE / 'reviewer-summary.json').write_text(json.dumps(summary, indent=2) + '\n', encoding='utf-8')
# Package text evidence only; do not upload complete datasets.
import zipfile
archive = REPO.parent / ('pr46-evidence-' + reviewed_commit[:12] + '.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for name in ['reviewer-summary.json', 'unit-tests.txt', 'data-verification-output.txt', 'data-verification.json']:
        bundle.write(EVIDENCE / name, arcname=name)
print(json.dumps(summary, indent=2))
print('Attach this ZIP to PR #46 with your own review note:', archive)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print('Outside Colab: retrieve the ZIP from the path above.')
